# 04 · Why did tech layoffs happen? Three hypotheses, three tests

Three explanations are usually given for the 2022-2023 layoff wave:

1. **Interest rates and the market (H1).** Money got expensive and tech valuations fell, so companies cut costs.
2. **Two dated shocks (H2).** The first Fed rate hike (2022-03-16) and the arrival of ChatGPT (2022-11-30) are often named as turning points.
3. **Over-hiring (H3).** Companies that expanded fastest in the 2020-2021 boom had the most to unwind.

For each one this notebook states the **test**, shows the **result** and states its **limit**. **Associations are not causes**: with one rate cycle, one wave of layoffs and a few dozen companies, the data can show whether a story is *consistent* with what happened, rarely that it is the reason.

Data: layoff tracker (US events), FRED (fed funds rate, NASDAQ), BLS JOLTS (information sector layoff rate), and the SEC panel from notebook 03.

In [ ]:
from pathlib import Path
import os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "build_master.py").exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)      # the project's data paths are relative to the repo root
from analysis import causes as cs

FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (10, 4.2), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 12, "axes.titleweight": "bold"})
def save(fig, name): fig.savefig(FIG / name, dpi=150, bbox_inches="tight")
def mark(ax):
    for d, lab in ((cs.FED_HIKE, "first Fed hike"), (cs.CHATGPT, "ChatGPT")):
        ax.axvline(d, color="k", ls="--", lw=1)
    return ax

data = cs.load_inputs()
events, fred, jolts = data["events"], data["fred"], data["jolts"]
us = events[events["region"] == "US"]
print(f"Tracker: {len(events):,} events, {len(us):,} in the US, {events['date'].min().date()} to {events['date'].max().date()}")
print("JOLTS available:", jolts is not None and len(jolts) > 0)

## 1 · The three series side by side

Dashed lines mark the two dates used in H2. The tracker starts in March 2020, so the first months are partial and are left out.

In [ ]:
h1 = cs.run_h1(events, fred, jolts)
lay, macro = h1["monthly"], h1["macro"]
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axes[0].bar(lay.index, lay["events"], width=25, color="#4C78A8"); axes[0].set_ylabel("US layoff events\nper month")
axes[1].plot(macro.index, macro["fedfunds"], color="#E45756"); axes[1].set_ylabel("Fed funds rate (%)")
axes[2].plot(macro.index, macro["nasdaq"], color="#54A24B"); axes[2].set_ylabel("NASDAQ Composite")
for ax in axes: mark(ax)
axes[2].set_xlim(lay.index.min(), lay.index.max())
axes[0].set_title("Layoffs, interest rates and the market")
save(fig, "04_overview.png"); plt.show()

## H1 · Do interest-rate and market changes come before layoffs?

**Test.** For each pair we correlate the 12-month change of the Fed funds rate (or of the NASDAQ, in logs) with a 3-month sum of layoff events, at lags from -12 to +12 months. A **positive lag means the macro series comes first**. The p-value is for the *largest* absolute correlation over all 25 lags. It asks how often the layoff series, moved around the calendar by every possible shift (circular shifts keep its autocorrelation and its hump), would match the macro series as well. With about 76 months there are 51 such shifts, so the smallest possible p-value is about 0.02.

In [ ]:
rows = []
for (x, y), r in h1["tests"].items():
    rows.append({"macro series": x, "layoff measure": y, "best lag (months)": r["best_lag"],
                 "correlation": round(r["best_corr"], 2), "p-value": round(r["p_value"], 3), "months": r["n_months"]})
h1_table = pd.DataFrame(rows); display(h1_table)

pairs = [k for k in h1["tests"] if k[1] == "tracker events (3-month sum)"]
fig, axes = plt.subplots(1, len(pairs), figsize=(10, 3.6), sharey=True)
for ax, k in zip(np.atleast_1d(axes), pairs):
    t = h1["tests"][k]["table"]
    ax.bar(t["lag_months"], t["corr"], color=np.where(t["lag_months"] > 0, "#4C78A8", "#BBBBBB"))
    ax.axhline(0, color="k", lw=0.8); ax.set_title(k[0].replace("_chg12", " (12-month change)"), fontsize=10)
    ax.set_xlabel("lag in months (positive = macro series first)")
np.atleast_1d(axes)[0].set_ylabel("correlation with layoff events")
save(fig, "04_h1_lead_lag.png"); plt.show()

**H1 result.**

- **How often companies announce layoffs follows rates and the market closely.** The 12-month change in the Fed funds rate has a correlation of +0.82 with the 3-month count of US layoff events (best lag +1 month, macro series first). The 12-month change in the NASDAQ has -0.85 (best lag +4 months; a falling market goes with more layoffs). In both cases no shifted version of the layoff series matched as well, so p = 0.019, the smallest value this test can give.
- **How many people were laid off does not show it.** With people laid off (3-month sum, log) as the measure, neither series is significant: the correlations are +0.44 (p about 0.6) and +0.65 (p about 0.4), and the NASDAQ best lag sits at -12, the edge of the range, so there is no clear lead.
- **The official layoff rate does not show it either.** The JOLTS layoff rate of the information sector (monthly, 128 months from 2015) has correlations of -0.21 with the Fed funds change (best lag -1 month) and +0.27 with the NASDAQ change (best lag -12), both with p about 0.6.
- **Reading.** The *number of announcements* rose when rates rose and the market fell. Neither the *size* of the cuts (dominated by a few very large announcements) nor the official layoff rate of the sector lines up. This is consistent with H1 for how often cuts are announced, not for how deep or how widespread they are.

**H1 limit.** There is one rate cycle and about six years of data, so almost any two trending series correlate; the test accounts for the 25 lags searched and for autocorrelation, but not for the fact that the rate rise and the layoff wave each happen once. A 12-month change describes the 12 months up to each date, so its curve trails the actual move; a best lag of +1 month does **not** mean rates acted one month before layoffs (the first hike came about eight months before the peak in announcements). The tracker also became far better known in 2022, so part of the surge in *recorded* events may be better coverage and not more layoffs. The JOLTS information sector also includes media and telecom, not only tech.

## H2 · Did layoffs jump after the first Fed hike, or after ChatGPT?

**Test.** For each date we compare the mean number of layoff events per week in the 26 weeks **after** with the 26 weeks **before**. To judge whether that jump is unusual, we compute the same before/after difference for **every other week in the tracker** (placebo dates) and report where the real date ranks. We add the JOLTS layoff rate of the information sector against all sectors, 6 months before vs after, as a second source.

In [ ]:
h2 = cs.run_h2(events, fred, jolts)
rows = []
for name, s in h2["studies"].items():
    rows.append({"event": name, "weekly events before": round(s["before_mean"], 2), "after": round(s["after_mean"], 2),
                 "difference": round(s["diff"], 2), "jump larger than at % of dates": round(100 * s["rank_share"]),
                 "largest jump in sample": str(s["largest_jump_date"].date())})
display(pd.DataFrame(rows))
print("Context (rate and market around each date):")
for name, c in h2["context"].items(): print(" ", name, {k: round(v, 2) for k, v in c.items()})
if h2["jolts"]:
    print("JOLTS layoff rate (% of employment), mean of 6 months before vs 6 months from the event month:")
    display(pd.DataFrame({(n, k): {"before": round(v["before"], 2), "after": round(v["after"], 2)}
                          for n, d in h2["jolts"].items() for k, v in d.items()}).T)

In [ ]:
wk = h2["weekly"]
fig, axes = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True)
axes[0].plot(wk.index, wk.rolling(4).mean(), color="#4C78A8"); axes[0].set_ylabel("US layoff events per week\n(4-week average)")
for name, d, col in (("first Fed hike", cs.FED_HIKE, "#E45756"), ("ChatGPT", cs.CHATGPT, "#F58518")):
    axes[0].axvspan(d - pd.Timedelta(weeks=26), d, color=col, alpha=0.10); axes[0].axvspan(d, d + pd.Timedelta(weeks=26), color=col, alpha=0.22)
    mark(axes[0])
pl = h2["studies"]["first Fed hike 2022-03-16"]["placebo"]
axes[1].plot(pl.index, pl, color="#777777"); axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_ylabel("after minus before\n(26 weeks each side)"); axes[1].set_title("The same before/after difference at every possible date")
mark(axes[1]); axes[0].set_title("Weekly layoff events and the two event windows (light = before, dark = after)")
save(fig, "04_h2_event_study.png"); plt.show()

**H2 result.**

- **The step up came in spring 2022, not at either date.** Weekly events averaged 1.0 in the 26 weeks before the first Fed hike and 15.4 in the 26 weeks after, a larger jump than at 95% of all dates. But the largest before/after jump in the whole sample is centred on the week of 2022-05-23 (+18.9), about ten weeks after the hike, once the hike was already priced into the market.
- **No jump at ChatGPT.** Average weekly events were 21.3 before 2022-11-30 and 21.6 after (difference +0.2, larger than only 67% of dates, so unremarkable). Layoffs were already running at about 21 events per week.
- **Context.** In the 26 weeks after the hike the Fed funds rate went from 0.08% to 2.33% and the NASDAQ fell 9.5%. After 2022-11-30 the NASDAQ rose 17.8% while rates went from 3.83% to 5.08%, and the weekly count stayed near 21 even as the market recovered.
- **The official data show no matching jump.** The JOLTS layoff rate of the information sector (mean of the 6 months before the event month against the 6 months from it) was 1.17% before and 1.15% after the first Fed hike, and 1.27% before and 1.38% after the ChatGPT date. For all sectors together it was 0.92% to 0.97% and 0.98% to 1.07%. So while recorded announcements went from about 1 to 15 a week, the sector's overall layoff rate hardly moved.
- **Reading.** The timing of the tracker fits the rate-and-valuation story as the start of the 2022 wave and does not fit ChatGPT as its trigger. But the flat official rate means the tracker's surge is not a measure of how many people the sector laid off: it largely reflects which cuts were announced and recorded (large, public, newsworthy companies). Information includes media and telecom, and one announcement can be spread over many months, so this does not prove that tech layoffs did not rise; it shows that the tracker alone cannot tell us how much.

**H2 limit.** A break in May 2022 could equally be the post-pandemic correction, the end of cheap funding, or the tracker becoming widely known; this test cannot separate those. The count of layoff events says nothing about whether AI changed *hiring*, or about later years, and the tracker has no field saying whether a layoff was linked to AI. A flat count after ChatGPT is therefore not evidence that AI played no role.

## H3 · Did companies that over-built in the boom lay off more later?

**Test.** For US-listed tech companies with four complete quarters of revenue and operating income in both 2019 and 2021:

- *boom growth* = ln(revenue 2021 / revenue 2019), *cost growth* = the same for total costs (revenue minus operating income), *over-build* = cost growth minus boom growth (costs grew faster than revenue);
- outcome = at least one tracker layoff in 2022 or 2023; companies already in the tracker before 2022 are left out so the outcome is not mixed with COVID-era layoffs.

We report each measure's AUC on its own (0.5 = no signal), then a logistic regression that also controls for **company size** (log assets at the end of 2019) and industry, because notebook 03 showed that size decides who appears in the tracker at all.

In [ ]:
h3 = cs.run_h3(data["fin"], data["revenue"], data["universe"], data["matches"], events)
u = h3["used"]
print(f"{len(h3['all'])} tech companies with complete 2019 and 2021 data; {h3['dropped_early_layoff']} dropped (layoff before 2022); "
      f"{len(u)} used, {int(u['y_layoff'].sum())} with a layoff in 2022-2023 ({u['y_layoff'].mean():.1%})")
print("\nAUC of each measure on its own (95% interval: bootstrap over companies)"); display(h3["auc"].round(3))
print("Size- and industry-controlled logistic regression, odds ratio per 1 standard deviation")
display(pd.concat({"over-build": h3["logit_overbuild"], "revenue and cost growth": h3["logit_both"]}).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, col, title in zip(axes, ["boom_growth", "overbuild"], ["revenue growth 2019-2021", "over-build (costs minus revenue growth)"]):
    t = h3["quartiles"][col]
    ax.bar(t["quartile"].astype(str), 100 * t["rate"], color="#4C78A8")
    for i, (r, n) in enumerate(zip(t["rate"], t["companies"])): ax.text(i, 100 * r + 0.5, f"n={n}", ha="center", fontsize=8)
    ax.set_title(title, fontsize=10); ax.tick_params(axis="x", labelrotation=20)
axes[0].set_ylabel("% with a tracker layoff in 2022-2023")
save(fig, "04_h3_overbuild.png"); plt.show()

**H3 result.** 347 tech companies had complete 2019 and 2021 data; 10 were dropped for a layoff before 2022, leaving 337, of which 47 (14%) had a tracker layoff in 2022-2023.

- **Fast growers more often appear in the tracker.** The share with a layoff rises steadily with revenue growth: 4.7%, 9.5%, 19.0% and 22.6% from the slowest to the fastest quarter of companies. The AUC is 0.65 (95% interval 0.57 to 0.74). Cost growth looks similar (7.1% to 23.8%, AUC 0.63).
- **Costs outrunning revenue is not the pattern.** Over-build (cost growth minus revenue growth) has an AUC of 0.51 (0.43 to 0.58) and no trend across quarters (11.8%, 14.3%, 20.2%, 9.5%). In the size-controlled regression its odds ratio is 1.08 (0.69 to 1.64).
- **After controlling for size and industry, cost growth still stands out.** Odds ratio 1.9 per standard deviation (1.2 to 2.7); revenue growth 1.2 (0.7 to 1.9). The two are strongly correlated, so the data cannot say which one matters; company size is stronger than both (odds ratio about 3.6).
- **Reading.** Companies that expanded fastest in the boom more often announced layoffs afterwards, even among companies of similar size. That fits a "boom, then correction" story. The specific version where costs outran revenue gets no support, but with 47 positives and a crude cost measure this is weak evidence against it, not strong evidence.

**H3 limit.** There is no headcount data: costs include acquisitions, stock compensation and cost of goods. Fast growers are often better-known companies, so being more visible to the tracker is an alternative explanation. With 47 positives the intervals are wide, and the 2019 and 2021 quarters are calendar-aligned approximations of fiscal quarters. The numbers above use the company matching of 2026-10-30 (350 matched layoff companies); an earlier, smaller matching gave the same picture.

## What the three tests can and cannot say together

1. **Timing matches the rates-and-market story, not the AI story.** Announcements surged in spring 2022 after the first Fed hike and during the NASDAQ decline, and there is no further step at ChatGPT. This is consistent with rates and valuations as the trigger of the wave; it does not show it.
2. **The tracker records how often companies announce cuts, not how many people the sector lost.** Event counts line up with rates and markets, but the number of people laid off does not, and the official JOLTS layoff rate of the information sector hardly moved while announcements rose fifteen-fold. Treat the tracker as a record of visible announcements.
3. **Companies that grew fastest in 2019-2021 were more likely to appear later**, beyond what their size explains. The narrower claim that they *over-hired relative to revenue* is not supported by the cost measure we have.
4. **What is still missing:** headcount data for H3 and any way to tell AI-related layoffs apart.